In [ ]:
!pip install requests

In [ ]:
!pip install atproto


In [ ]:
import os
from atproto import Client

email = os.environ["AAPAD_BLUESKY_EMAIL"]
app_password = os.environ["AAPAD_BLUESKY_APP_PASSWORD"]
client = Client()
client.login(email, app_password)

In [ ]:
search_results = client.app.bsky.feed.search_posts(params={'q': 'floods in india', 'limit': 10})

for post in search_results.posts:
    print(f"Author: {post.author.handle}")
    print(f"Text: {post.record.text}")
    print(f"URI: {post.uri}")
    print("---------------------")

In [ ]:
!pip install transformers

In [ ]:
from transformers import pipeline

# Load a pre-trained sentiment analysis model
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

positive_posts = []

print("Performing sentiment analysis and filtering for positive posts...")
for post in search_results.posts:
    text = post.record.text
    if text:
        result = sentiment_analyzer(text)[0]
        # The model typically returns 'POSITIVE' or 'NEGATIVE'
        # and a score. We'll filter for 'POSITIVE' sentiments.
        if result['label'] == 'POSITIVE' and result['score'] > 0.8:
            positive_posts.append({
                'author': post.author.handle,
                'text': text,
                'uri': post.uri,
                'sentiment_label': result['label'],
                'sentiment_score': result['score']
            })

print(f"Found {len(positive_posts)} positive posts.\n")

if positive_posts:
    print("Posts with Positive Sentiment (score > 0.8):")
    for p_post in positive_posts:
        print(f"Author: {p_post['author']}")
        print(f"Text: {p_post['text']}")
        print(f"URI: {p_post['uri']}")
        print(f"Sentiment: {p_post['sentiment_label']} (Score: {p_post['sentiment_score']:.2f})")
        print("---------------------")
else:
    print("No posts with strong positive sentiment found among the search results.")